In [1]:
import re
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

c:\Users\T14s\Arix-Research-Paper-Recommendation-System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Locate the repo from the current directory, so the notebook runs from anywhere.
HERE = Path.cwd()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError(f"Start the notebook inside the repo: no pyproject.toml above {HERE}.")
DATA_DIR = ROOT / "data"

In [3]:
def ensure_nltk(pkg):
    """Download an NLTK package only when it is missing; fail with context if that fails."""
    for name in (f"corpora/{pkg}", f"corpora/{pkg}.zip"):
        try:
            nltk.data.find(name)
            return
        except LookupError:
            pass
    try:
        nltk.download(pkg, raise_on_error=True)
    except Exception as e:
        raise RuntimeError(
            f"NLTK data '{pkg}' is missing and could not be downloaded. "
            "Check your network or pre-install it (see nltk.data.path)."
        ) from e

In [4]:
for _pkg in ("stopwords", "wordnet"):
    ensure_nltk(_pkg)

In [5]:
df = pd.read_csv(DATA_DIR / "arxiv_papers_v2.csv")

In [6]:
df.head()

,request_id,retrieved_at,domain,arxiv_id,title,summary,authors,primary_category,categories,published,updated,abs_url,pdf_url
0,https://arxiv.org/api/Nn9ZAl7r/hPtw2SqI662/Ri9DWc,2026-10-30T05:02:17+00:00,cs,2610.40655v1,Elucidating the Design Space of LM Data Attrib...,Data attribution methods for Language Models (...,Steven Yu; Sheng-Yu Wang; Kimia Hamidieh; Anto...,cs.CL,cs.CL,2026-10-29T17:59:53Z,2026-10-29T17:59:53Z,http://arxiv.org/abs/2610.40655v1,https://arxiv.org/pdf/2610.40655v1
1,https://arxiv.org/api/Nn9ZAl7r/hPtw2SqI662/Ri9DWc,2026-10-30T05:02:17+00:00,cs,2610.40654v1,Propose-Filter-Release: Scaling Noise to Typic...,An estimator can be accurate and stable on typ...,Moshe Shenfeld; Eliad Tsfadia; Katrina Ligett,cs.LG,cs.LG,2026-10-29T17:59:49Z,2026-10-29T17:59:49Z,http://arxiv.org/abs/2610.40654v1,https://arxiv.org/pdf/2610.40654v1
2,https://arxiv.org/api/Nn9ZAl7r/hPtw2SqI662/Ri9DWc,2026-10-30T05:02:17+00:00,cs,2610.40653v1,Micro-Edits: Computational Support for Fine-Sc...,"In film editing, every frame matters. While ma...",Nhan Tran; Hijung Valentina Shin; Abe Davis; M...,cs.HC,cs.HC; cs.CV,2026-10-29T17:59:49Z,2026-10-29T17:59:49Z,http://arxiv.org/abs/2610.40653v1,https://arxiv.org/pdf/2610.40653v1
3,https://arxiv.org/api/Nn9ZAl7r/hPtw2SqI662/Ri9DWc,2026-10-30T05:02:17+00:00,cs,2610.40649v1,Rethinking Neural Thickets Without Weight Pert...,The Neural Thickets hypothesis posits that pre...,Nicolas Dufour; Elouan Gardès; Amil Dravid; Pa...,cs.LG,cs.LG; cs.AI; cs.CV,2026-10-29T17:59:29Z,2026-10-29T17:59:29Z,http://arxiv.org/abs/2610.40649v1,https://arxiv.org/pdf/2610.40649v1
4,https://arxiv.org/api/Nn9ZAl7r/hPtw2SqI662/Ri9DWc,2026-10-30T05:02:17+00:00,cs,2610.40647v1,"Read, Allocate, Route: Role-Coupled Tokenizati...",Pixel-space generative Transformers typically ...,Jiongfan Yang; Weichen Fan; Haiwen Diao; Zhong...,cs.CV,cs.CV; cs.AI; cs.LG; cs.NE,2026-10-29T17:59:21Z,2026-10-29T17:59:21Z,http://arxiv.org/abs/2610.40647v1,https://arxiv.org/pdf/2610.40647v1


In [7]:
df.tail()

,request_id,retrieved_at,domain,arxiv_id,title,summary,authors,primary_category,categories,published,updated,abs_url,pdf_url
1995,https://arxiv.org/api/Y+42imjzfw303xla7f5OMcIBc1c,2026-10-30T05:02:17+00:00,eess,2610.24651v1,Target Sensing in Full-Dimensional MIMO-OFDM I...,Integrated sensing and communication (ISAC) ha...,Ruoyu Zhang; Tingjie Yang; Tianyu Zhao; Lei Ch...,eess.SP,eess.SP,2026-10-17T11:54:57Z,2026-10-17T11:54:57Z,http://arxiv.org/abs/2610.24651v1,https://arxiv.org/pdf/2610.24651v1
1996,https://arxiv.org/api/Y+42imjzfw303xla7f5OMcIBc1c,2026-10-30T05:02:17+00:00,eess,2610.24645v1,Distributed Real-Time Cooperative Model Predic...,This paper presents a novel distributed real-t...,Yuning Jiang; Kristína Fedorová; Roland Schwan...,eess.SY,eess.SY,2026-10-17T11:45:40Z,2026-10-17T11:45:40Z,http://arxiv.org/abs/2610.24645v1,https://arxiv.org/pdf/2610.24645v1
1997,https://arxiv.org/api/Y+42imjzfw303xla7f5OMcIBc1c,2026-10-30T05:02:17+00:00,eess,2610.24641v1,Widely Linear-Enhanced IQ Imbalance Interferen...,Internet of Things (IoT) networks require reli...,Xianle Dai; Bowei Xue; Qu Luo; Jianguo Li; Nen...,eess.SP,eess.SP,2026-10-17T11:32:54Z,2026-10-17T11:32:54Z,http://arxiv.org/abs/2610.24641v1,https://arxiv.org/pdf/2610.24641v1
1998,https://arxiv.org/api/Y+42imjzfw303xla7f5OMcIBc1c,2026-10-30T05:02:17+00:00,eess,2610.24622v1,Fluxion: A Differentiable DSP Runtime from Tra...,Native real-time audio engines meet audio dead...,Matteo Spanio; Antonio Rodà,eess.SP,eess.SP; cs.SD; eess.AS,2026-10-17T10:56:08Z,2026-10-17T10:56:08Z,http://arxiv.org/abs/2610.24622v1,https://arxiv.org/pdf/2610.24622v1
1999,https://arxiv.org/api/Y+42imjzfw303xla7f5OMcIBc1c,2026-10-30T05:02:17+00:00,eess,2610.24599v1,Severity-Channel Conditioning: An Injection-Le...,Motion-deblurring models lose substantial accu...,Abdullah Al Shafi; Kazi Saeed Alam; Sumaiya Ra...,eess.IV,eess.IV; cs.AI; cs.CV; cs.LG; cs.MM,2026-10-17T10:05:03Z,2026-10-17T10:05:03Z,http://arxiv.org/abs/2610.24599v1,https://arxiv.org/pdf/2610.24599v1


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   request_id        2000 non-null   str  
 1   retrieved_at      2000 non-null   str  
 2   domain            2000 non-null   str  
 3   arxiv_id          2000 non-null   str  
 4   title             2000 non-null   str  
 5   summary           2000 non-null   str  
 6   authors           2000 non-null   str  
 7   primary_category  2000 non-null   str  
 8   categories        2000 non-null   str  
 9   published         2000 non-null   str  
 10  updated           2000 non-null   str  
 11  abs_url           2000 non-null   str  
 12  pdf_url           2000 non-null   str  
dtypes: str(13)
memory usage: 203.3 KB


In [9]:
for col in ("retrieved_at", "published", "updated"):
    df[col] = pd.to_datetime(df[col], format="ISO8601", utc=True, errors="raise")

In [10]:
assert not df[["title", "summary"]].apply(lambda s: s.str.strip().eq("")).any().any(), "blank title/summary"

In [11]:
df["text"] = df["title"].str.strip() + ". " + df["summary"].str.strip()

In [12]:
### Dataset Preprocessing
wnl = WordNetLemmatizer()
def preprocessing(text):
    text_clean = re.sub('[^a-zA-Z]', ' ', text)
    text_clean= text_clean.lower()
    text_clean = text_clean.split()

    sw = set(stopwords.words("english"))

    text_clean = [wnl.lemmatize(word) for word in text_clean if not word in sw]
    text_final = " ".join(text_clean)
    return text_final

In [13]:
tqdm.pandas()

df["text_clean"] = df["text"].progress_apply(preprocessing)

100%|██████████| 2000/2000 [00:04<00:00, 450.55it/s] 


In [14]:
model = SentenceTransformer("BAAI/bge-base-en-v1.5")
df["text_clean_embeddings"] = list(
    model.encode(df["text_clean"].tolist(), batch_size=32, show_progress_bar=True)
)

Batches: 100%|██████████| 63/63 [07:30<00:00,  7.16s/it]


In [15]:
df[["text_clean", "text_clean_embeddings"]].head()

,text_clean,text_clean_embeddings
0,elucidating design space lm data attribution d...,"[0.023616, 0.013599, -0.027838, 0.040844, 0.07..."
1,propose filter release scaling noise typical s...,"[-0.008773, 0.012464, 0.023553, 0.02183, 0.051..."
2,micro edits computational support fine scale e...,"[0.01309, 0.038795, -0.001134, 0.032718, 0.052..."
3,rethinking neural thicket without weight pertu...,"[0.007173, -0.072364, -0.011465, 0.007351, 0.0..."
4,read allocate route role coupled tokenization ...,"[-0.01938, -0.037332, -0.00891, 0.040403, 0.01..."


In [ ]:
df.drop(columns="text_clean_embeddings").to_csv(
    DATA_DIR / "arxiv_papers_preprocessed_v2.csv", index=False
)
np.save(DATA_DIR / "arxiv_embeddings_v2.npy", np.vstack(df["text_clean_embeddings"]))